# Caminhos mínimos — versão didática

Notebook enxuto para estudar e comparar:
- Algoritmo de Dijkstra (ótimo)
- Uma heurística gulosa de vizinho mais próximo (não garante ótimo)

Foco: simplicidade, contratos claros e um exemplo executável.

## Objetivos
- Gerar um grafo completo dirigido com pesos positivos.
- Implementar Dijkstra com contagem de comparações.
- Implementar uma heurística gulosa simples.
- Comparar resultados em um exemplo pequeno.

In [ ]:
from typing import Dict, List, Tuple, Optional
from collections import deque
from math import inf
import heapq
import random

## Geração de grafo completo
Contrato:
- Entrada: n (vértices), faixa de pesos e seed opcional.
- Saída: dicionário {u: [(v, w), ...]} sem laços (u != v).

In [ ]:
def grafo_completo(n: int, wmin: float = 1.0, wmax: float = 10.0, seed: Optional[int] = None) -> Dict[int, List[Tuple[int, float]]]:
    if seed is not None:
        random.seed(seed)
    adj: Dict[int, List[Tuple[int, float]]] = {u: [] for u in range(n)}
    for u in range(n):
        for v in range(n):
            if u != v:
                adj[u].append((v, random.uniform(wmin, wmax)))
    return adj

## Algoritmo de Dijkstra
- Garante caminhos mínimos (pesos não-negativos).
- Usa fila de prioridade (heap).
- Complexidade típica: O((V + E) log V).
- Contamos uma comparação a cada tentativa de relaxamento.

In [ ]:
def dijkstra(adj: Dict[int, List[Tuple[int, float]]], s: int = 0):
    if not adj or s not in adj:
        raise ValueError("Grafo vazio ou origem inválida")
    for u, viz in adj.items():
        for v, w in viz:
            if w < 0:
                raise ValueError(f"Dijkstra requer pesos não negativos (aresta {u}->{v} tem peso {w})")
    dist: Dict[int, float] = {u: inf for u in adj}
    parent: Dict[int, Optional[int]] = {u: None for u in adj}
    dist[s] = 0.0
    pq: List[Tuple[float, int]] = [(0.0, s)]
    comparisons = 0
    while pq:
        du, u = heapq.heappop(pq)
        if du != dist[u]:
            continue
        for v, w in adj[u]:
            comparisons += 1
            nd = du + w
            if nd < dist[v]:
                dist[v] = nd
                parent[v] = u
                heapq.heappush(pq, (nd, v))
    return dist, parent, comparisons

## Heurística gulosa (vizinho mais próximo)
- Não garante soluções ótimas.
- Simples e rápida na prática.
- Usamos deque para remoção O(1).

In [ ]:
def gulosa_vizinho_proximo(adj: Dict[int, List[Tuple[int, float]]], s: int = 0):
    if s not in adj:
        raise ValueError("Origem inválida")
    dist: Dict[int, float] = {u: inf for u in adj}
    parent: Dict[int, Optional[int]] = {u: None for u in adj}
    dist[s] = 0.0
    visited = {s}
    q = deque([s])
    comparisons = 0
    while q:
        u = q.popleft()
        melhor = None
        melhor_custo = inf
        for v, w in adj[u]:
            comparisons += 1
            if v in visited:
                continue
            custo = dist[u] + w
            if custo < dist[v] and custo < melhor_custo:
                melhor_custo = custo
                melhor = v
        if melhor is not None:
            dist[melhor] = melhor_custo
            parent[melhor] = u
            visited.add(melhor)
            q.append(melhor)
    return dist, parent, comparisons

## Reconstrução de caminho
Retorna a lista de vértices do caminho da origem até o destino.

In [ ]:
def caminho(parent: Dict[int, Optional[int]], t: int) -> List[int]:
    rota = []
    while t is not None:
        rota.append(t)
        t = parent[t]
    return list(reversed(rota))

## Exemplo rápido
Geramos um grafo pequeno, rodamos Dijkstra e a heurística, e comparamos as distâncias até um destino.

In [ ]:
n = 6
g = grafo_completo(n, seed=42)
dist_d, parent_d, comp_d = dijkstra(g, 0)
dist_g, parent_g, comp_g = gulosa_vizinho_proximo(g, 0)
print(f'Comparações — Dijkstra: {comp_d:,} | Gulosa: {comp_g:,}')
t = 5
print(f'Distância até {t} — Dijkstra: {dist_d[t]:.2f} | Gulosa: {dist_g[t]:.2f}')
print('Caminho Dijkstra:', caminho(parent_d, t))
print('Caminho Gulosa  :', caminho(parent_g, t))